# 10 — Multi-Agent Orchestration (real LangGraph, real GRPO)

`agenttune.agentic.langgraph_orchestrator.AgentTuneGraph` composes a rollout node and LLM-judge
nodes into one graph, compiling drop-in callables for a trainer:

```python
graph = AgentTuneGraph()
graph.add_rollout("agent", rollout_fn)
graph.add_judge("quality", judge_a); graph.add_judge("strict", judge_b, aggregation="min")
trainer = create_agentic_trainer("grpo", model=..., train_dataset=ds,
    rollout_func=graph.compile_rollout(), reward_funcs=[graph.compile_grpo_reward()])
```

This notebook runs that documented one-liner for real: a live `Qwen2.5-3B-Instruct` agent (with
a real `calc` tool) is the rollout node; **two** real, deliberately-different `LLMJudge`s (also
live Qwen2.5-3B — one lenient, one harsh) score each rolled-out trajectory, mean-aggregated per
completion, driving a real GRPO LoRA `train()`.

**Honest scope**: the claim is that a real, non-constant, graph-composed judge reward drives a
real `GRPOTrainer.train()` — not a convergence claim. The math problems below are hand-picked to
span easy -> hard (disclosed, not hidden) so the two judges produce a genuine *spread* of scores
rather than everything flooring to 1.0 — the same principle notebook 01 used filtering GSM8K by
difficulty, just applied to picking problems instead of filtering a dataset.

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://huggingface.co"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

import sys
sys.modules["vllm"] = None  # this env's vllm wheel is ABI-incompatible with torch

import statistics
import torch
from datasets import Dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM

assert torch.cuda.is_available()
print(f"[gpu] {torch.cuda.get_device_name(0)} | torch {torch.__version__}")

MODEL = "Qwen/Qwen2.5-3B-Instruct"
OUT_DIR = "./_runs/11_langgraph_grpo"
os.makedirs(OUT_DIR, exist_ok=True)

[gpu] NVIDIA GeForce RTX 4090 | torch 2.11.0+cu130


## Two deliberately different real judges + a real tool-using rollout node

Lenient (answer-only) vs. strict (demands shown work + units) — so mean-aggregation is
observable rather than a no-op.

In [2]:
from agenttune.agentic.langgraph_orchestrator import AgentTuneGraph, make_grpo_rollout_func
from agenttune.agentic.rewards.llm_judge import LLMJudge
from agenttune.agentic.rollout_engines.rollout_factory import create_rollout_engine, create_rollout_fn
from agenttune.core.backend_factory import create_agentic_trainer

JUDGE_SYS_LENIENT = ('You grade a math answer for the FINAL NUMBER only. 1.0 if the '
                    'final number is correct (ignore missing units/work), ~0.5 if close/'
                    'rounded, 0.0 if wrong. Reply JSON: {"score": <float 0-1>, "explanation": "<short>"}')
JUDGE_SYS_STRICT = ('You are a HARSH grader. Give 1.0 ONLY if the final number is correct '
                   'AND the units are shown AND the working is shown. Deduct heavily '
                   '(cap at 0.4) for any missing units, rounding, or unshown steps; 0.0 if '
                   'the answer is wrong. Reply JSON: {"score": <float 0-1>, "explanation": "<short>"}')

# Deliberately spans easy -> hard (disclosed) so the judges produce a real spread of scores.
EXAMPLES = [
    {"prompt": "A bat and a ball cost $1.10 total; the bat costs $1.00 more than "
               "the ball. How much is the ball, in cents?"},                          # trick
    {"prompt": "Compound interest on $1000 at 10% per year for 3 years, "
               "compounded annually -- final amount?"},                                # hard
    {"prompt": "Area of an 8 m by 12 m rectangle?"},                                   # easy
    {"prompt": "What is 15% of 240?"},                                                 # easy
    {"prompt": "Convert 100 km to miles (1 km = 0.621371 miles)."},                    # medium
    {"prompt": "Simple interest on $1000 at 5% per year for 3 years?"},                # medium
]

def calculator(expression: str) -> dict:
    """Evaluate a mathematical expression.

    Args:
        expression: Python math expression string (e.g. '2 + 2').
    """
    try:
        return {"result": eval(expression, {"__builtins__": {}}, {})}
    except Exception as e:
        return {"error": str(e)}

def build_graph():
    engine = create_rollout_engine(backend="transformers", model_path=MODEL)
    rollout_fn = create_rollout_fn(
        rollout_engine=engine, tools=[calculator], max_steps=3,
        system_prompt="You are a math assistant. Use the calculator tool, then answer concisely.",
    )
    judge_a = LLMJudge(backend="transformers", model_path=MODEL, system_prompt=JUDGE_SYS_LENIENT, cache_size=0)
    judge_b = LLMJudge(backend="transformers", model_path=MODEL, system_prompt=JUDGE_SYS_STRICT, cache_size=0)

    graph = AgentTuneGraph()
    graph.add_rollout("agent", rollout_fn)
    graph.add_judge("lenient", judge_a, criteria={"correctness": 1.0}, aggregation="mean")
    graph.add_judge("strict", judge_b, criteria={"correctness": 1.0}, aggregation="mean")
    return graph

graph = build_graph()
print("[graph] real AgentTuneGraph: 1 tool-using rollout node + 2 real LLM-judge nodes")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[LLMJudge/transformers] Loading Qwen/Qwen2.5-3B-Instruct ...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[LLMJudge/transformers] Ready.
[LLMJudge/transformers] Loading Qwen/Qwen2.5-3B-Instruct ...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[LLMJudge/transformers] Ready.
[graph] real AgentTuneGraph: 1 tool-using rollout node + 2 real LLM-judge nodes


## Preflight — the graph-composed judge reward is real and discriminating

Runs the real rollout + both real judges over the batch and checks: rewards aren't a constant
0.5 stub, at least one score is genuinely parsed (>0), and the two judges actually diverge.

In [3]:
compiled = graph.compile_rollout()
grpo_reward = graph.compile_grpo_reward()

pre = compiled([e["prompt"] for e in EXAMPLES], temperature=0.9, top_p=0.95)
jr = pre["judge_rewards"]
print(f"[preflight] per-completion judge_rewards: {[round(x, 3) for x in jr]}")
js = pre["judge_scores"]
print(f"[preflight] lenient={js['lenient']:.3f}  strict={js['strict']:.3f}  "
     f"delta={abs(js['lenient']-js['strict']):.3f}")

assert max(jr) > 0.0, "every judge reward is 0 -- the judge never parsed"
assert len({round(x, 3) for x in jr}) >= 2, "judge rewards constant -- no better than a 0.5 stub"
assert abs(js["lenient"] - js["strict"]) > 1e-6, "the two judges scored identically -- composition is a no-op"
print("[preflight] PASSED: real, non-constant, genuinely-divergent judge reward confirmed")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[preflight] per-completion judge_rewards: [0.0, 1.0, 1.0, 1.0, 0.75, 1.0]
[preflight] lenient=0.750  strict=0.833  delta=0.083
[preflight] PASSED: real, non-constant, genuinely-divergent judge reward confirmed


## The documented one-liner: graph rollout + graph reward -> real `GRPOTrainer.train()`

In [4]:
seen = {"batches": [], "all": []}

def recording_reward(completions=None, **kwargs):
    scores = grpo_reward(completions=completions, **kwargs)
    seen["batches"].append([round(float(s), 3) for s in scores])
    seen["all"].extend(float(s) for s in scores)
    return scores
recording_reward.__name__ = "agenttune_graph_reward"

ds = Dataset.from_list(EXAMPLES)
trainer = create_agentic_trainer(
    "grpo", model=MODEL, train_dataset=ds,
    rollout_func=make_grpo_rollout_func(compiled),
    reward_funcs=[recording_reward], output_dir=OUT_DIR,
    num_generations=2, per_device_train_batch_size=4, gradient_accumulation_steps=1,
    max_steps=4, max_completion_length=200, shuffle_dataset=False,
    temperature=0.9, learning_rate=1e-5, beta=0.04, seed=0, logging_steps=1, report_to="none",
    peft_config={"r": 16, "lora_alpha": 32, "lora_dropout": 0.0, "bias": "none",
                "task_type": "CAUSAL_LM", "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"]},
)
print("[train] real GRPO (LoRA) with the graph-composed rollout + judge reward...")
trainer.train()
gt = trainer.trainer
gt.save_model(OUT_DIR)
print("[train] done")

[train] real GRPO (LoRA) with the graph-composed rollout + judge reward...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,-0.065299
2,0.000000
3,0.000025
4,0.006438


[train] done


## Real evaluation: did the graph's judge reward actually score training completions?

Not a hand-rolled metric — the recorded reward values the real GRPOTrainer actually trained on,
plus the trainer's own `reward_std`/`grad_norm` log history.

In [5]:
all_r = seen["all"]
distinct = len({round(x, 3) for x in all_r})
print(f"[eval] completions scored: {len(all_r)}  distinct reward values: {distinct}")
print(f"[eval] any strictly in (0,1): {any(0 < x < 1 for x in all_r)}")
print(f"[eval] constant-0.5 stub?: {all(x == 0.5 for x in all_r)}")

rstds = [h["reward_std"] for h in gt.state.log_history if "reward_std" in h]
gnorms = [h["grad_norm"] for h in gt.state.log_history if "grad_norm" in h]
print(f"[eval] per-step reward_std: {[round(x,3) for x in rstds]}")
print(f"[eval] per-step grad_norm:  {[round(x,3) for x in gnorms]}")

adapter_ok = os.path.exists(os.path.join(OUT_DIR, "adapter_model.safetensors"))
reload_ok = False
if adapter_ok:
    base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to("cuda")
    reload_ok = isinstance(PeftModel.from_pretrained(base, OUT_DIR), PeftModel)

print(f"\n[verdict] graph rollout + graph reward drove GRPOTrainer.train(): True")
print(f"[verdict] judge reward non-constant (not a 0.5 stub): {distinct > 1}")
print(f"[verdict] judge genuinely parsed (max reward > 0): {max(all_r) > 0}")
print(f"[verdict] within-group reward variance (mean std, measured not gated): "
     f"{statistics.mean(rstds) if rstds else 0.0:.3f}")
print(f"[verdict] LoRA adapter saved + reloadable: {adapter_ok and reload_ok}")
print("[verdict] convergence: NOT claimed")

[eval] completions scored: 16  distinct reward values: 4
[eval] any strictly in (0,1): True
[eval] constant-0.5 stub?: False
[eval] per-step reward_std: [0.473, 0.0, 0.144, 0.377]
[eval] per-step grad_norm:  [0.289, 0.0, 0.03, 0.792]


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


[verdict] graph rollout + graph reward drove GRPOTrainer.train(): True
[verdict] judge reward non-constant (not a 0.5 stub): True
[verdict] judge genuinely parsed (max reward > 0): True
[verdict] within-group reward variance (mean std, measured not gated): 0.249
[verdict] LoRA adapter saved + reloadable: True
[verdict] convergence: NOT claimed


## Summary

| Check | Real evidence |
|---|---|
| Preflight | two real judges genuinely diverge, reward non-constant |
| Training | real `GRPOTrainer.train()` ran with the graph-composed rollout+reward |
| Reward source | recorded values are what the trainer actually trained on, not a re-derivation |
| Adapter | real LoRA weights saved and reloadable as a `PeftModel` |

The graph, both judges, and the trainer are all real — a live `Qwen2.5-3B-Instruct` plays every
role (agent, lenient judge, strict judge) — no constant-reward stub anywhere in the loop.